In [ ]:
import json
import re
from collections import defaultdict
from pathlib import Path

import mne
import numpy as np
import pandas as pd
from scipy.stats import kurtosis, zscore
from tqdm import tqdm

# Methods



In [ ]:

def kurtosis_matrix(session_raw, trial_onsets, trial_offsets, chs=None):
    """Return trial-by-channel kurtosis values for a cropped raw session."""
    if chs is None:
        chs = np.array(session_raw.ch_names)

    kurt_values = []
    for onset, offset in zip(trial_onsets, trial_offsets):
        trial_data = session_raw.get_data(picks=chs, tmin=onset, tmax=offset)
        kurt_values.append(kurtosis(trial_data, axis=-1))

    return np.array(kurt_values)


def kurtosis_rejection(kurt_matrix, threshold=2.5, axis=0):
    """Flag channels or trials whose average kurtosis z-score exceeds threshold."""
    # kurt_matrix shape: n_trials_or_sessions x n_channels
    avg_kurtosis = np.mean(kurt_matrix, axis=axis)
    z_kurtosis = zscore(avg_kurtosis)
    return z_kurtosis > threshold

In [3]:
def compute_event_tfr(
    bipolar,
    event_onsets,
    tmin,
    tmax,
    freqs,
    n_cycles,
    buffer_sec=5,
    baseline=None
):
    """
    Compute event-related time-frequency power using Morlet wavelets.

    Parameters
    ----------
    bipolar : mne.io.Raw
        Preprocessed raw data (downsampled, notch filtered, bipolar signal).
    event_onsets : array-like
        Event onset times in seconds.
    tmin, tmax : float
        Time window around event.
    freqs : array-like
        Frequencies for Morlet transform.
    n_cycles : float | array
        Number of cycles for each frequency.
    buffer_sec : float
        Padding to avoid wavelet edge effects.
    baseline : None | tuple of length 2
        The time interval to consider as “baseline” when applying baseline correction. Used in mne.Epochs
    
    Returns
    -------
    tfr : mne.time_frequency.EpochsTFR
        The time-frequency-resolved power estimates.
    """

    raw = bipolar.copy()

    # create annotations
    annot = mne.Annotations(
        onset=event_onsets,
        duration=0,
        description="event"
    )

    raw.set_annotations(annot, verbose='ERROR')

    events, event_id = mne.events_from_annotations(raw)

    epochs = mne.Epochs(
        raw,
        events,
        event_id,
        tmin=tmin - buffer_sec,
        tmax=tmax + buffer_sec,
        baseline=baseline,
        preload=True,
        verbose="ERROR",
        on_missing="raise",
        event_repeated="error"
    )

    tfr = mne.time_frequency.tfr_morlet(
        epochs,
        freqs=freqs,
        n_cycles=n_cycles,
        return_itc=False,
        average=False,
        output="power",
        n_jobs=-1,
        verbose="ERROR"
    )

    # remove buffer
    tfr.crop(tmin=tmin, tmax=tmax)

    return tfr.data

# Configuration and Main Workflow



In [ ]:

# Data configuration. Update these values when running the notebook on a new machine.
data_dir = Path(r"D:\Data\Sharing\RoT\DataSample")
subject = "Subj01"
SID = "S01"

# Processed FIF files used in the Morlet TFR section.
fif_pattern = f"{subject}_Session[1-9]_ieeg.fif.gz"
fif_files = sorted(data_dir.joinpath(subject).glob(fif_pattern))

TTL_params = np.load(data_dir / subject / f"{subject}_TTL.npz", allow_pickle=True)["TTL_dict"].item()

with open(data_dir / subject / f"{subject}_epileptic_channels.json", "r", encoding="utf-8") as f:
    params = json.load(f)
ied_chs = params[SID]["EP_chname"]

anat_df = pd.read_csv(data_dir / subject / f"{subject}_anat.csv")
bp_chs = anat_df[
    anat_df.tissues_segment.isin(["Gray", "White", "gray", "white"])
].ch_name.values

## Channel selection

In [ ]:
pick_chs = [ch for ch in bp_chs if ch not in ied_chs]

In [4]:
edf_to_session = defaultdict(list)
for session, ttl in TTL_params.items():
    edf_fn = data_dir / subject / ttl['filename']
    edf_to_session[edf_fn].append(session)

In [ ]:

channel_kurt = []
for edf_fn, sessions in edf_to_session.items():
    raw = mne.io.read_raw_edf(edf_fn, preload=False, verbose="ERROR")
    raw.pick(pick_chs, verbose="ERROR")

    for session in tqdm(sessions):
        session_ttl = TTL_params[session]
        tmin = session_ttl["Session_start_time"] - session_ttl["buffer_sec"]
        tmax = session_ttl["Session_stop_time"] + session_ttl["buffer_sec"]

        data = raw.copy().crop(tmin=tmin, tmax=tmax, verbose="ERROR")
        data = raw.resample(sfreq=1000, n_jobs=-1, verbose="ERROR")
        data.notch_filter(freqs=[50.0, 100.0, 150.0, 200.0, 250.0], n_jobs=-1, verbose="ERROR")

        # data.save(save_path / f"{subject}_Session{session}_ieeg.fif.gz", picks=pick_chs, overwrite=False, verbose="ERROR")
        kurt_values = kurtosis_matrix(
            data,
            session_ttl["TrialStart"],
            session_ttl["TrialStop"],
            chs=pick_chs,
        )
        channel_kurt.append(kurt_values)

        del data, kurt_values
    raw.close()

100%|██████████| 3/3 [08:07<00:00, 162.48s/it]


In [ ]:

reject_mask = kurtosis_rejection(np.concatenate(channel_kurt, axis=0), threshold=2.5, axis=0)
auto_noisy_chs = np.array(pick_chs)[reject_mask]
nosiy_chs = auto_noisy_chs  # Backward-compatible alias for the original notebook variable.
auto_noisy_chs

array(['POL K2', 'POL K3', 'POL K4'], dtype='<U12')

## Morlet TFR

In [ ]:

freqs = np.array([3, 5, 8, 12, 19, 31, 40, 79, 130, 210])
n_cycles = freqs / 2

# Replace this manual list with `auto_noisy_chs` above after inspecting the rejection output.
noisy_chs = ["POL K2", "POL K3", "POL K4"]
pick_chs = [ch for ch in bp_chs if ch not in ied_chs and ch not in noisy_chs]

event_dict = {
    "Movie": dict(label="VideoStart", tmin=0, tmax=4),
    "TOJ": dict(label="TOJStop", tmin=-2, tmax=0),
    "Rest": dict(label="TOJStop", tmin=0, tmax=5),
}


In [ ]:

for fif in fif_files:
    fif = Path(fif)
    session = re.search(r"Session\d+", fif.name).group(0)

    raw = mne.io.read_raw_fif(fif, preload=False, verbose="ERROR")  # downsampled, notched
    raw.pick(pick_chs, verbose="ERROR")
    raw.load_data(verbose="ERROR")
    raw.set_eeg_reference(ref_channels="average", projection=False, verbose="ERROR")

    TTL_onsets = TTL_params.get(session)
    for event, event_kws in tqdm(event_dict.items()):
        event_onsets = TTL_onsets[event_kws["label"]]
        tmin, tmax = event_kws["tmin"], event_kws["tmax"]

        event_tfr = compute_event_tfr(
            raw,
            event_onsets,
            tmin,
            tmax,
            freqs,
            n_cycles,
            buffer_sec=5,
            baseline=None,
        )
        log_power = np.log(event_tfr)

        # np.savez_compressed(output_path, data=log_power)
    raw.close()
    del event_tfr, log_power